# 감정 모델 추가 학습 (KEMDy19)
`HyunaZ/hubert_emotion`을 KEMDy19로 이어서 학습합니다. **런타임 → 런타임 유형 변경 → GPU(T4)** 로 먼저 바꾸세요.

준비물: ETRI NANUM에서 KEMDy19를 받아(라이선스 동의 필요) 압축 파일 그대로 **Google Drive 내 드라이브**에 `KEMDy19.zip`으로 올려 두기.
위에서부터 한 칸씩 실행하면 됩니다.

In [ ]:
# 1) GPU 확인 — 'Tesla T4' 같은 표가 나와야 합니다
!nvidia-smi

In [ ]:
# 2) 코드 받기 (공개 저장소)
!git clone https://github.com/jaeuk6797-cell/kuccton.git
%cd kuccton
!git checkout feat/finetune-kemdy19

In [ ]:
# 3) 라이브러리 설치 (torch는 Colab에 이미 있습니다)
!pip -q install transformers pytorch-lightning librosa huggingface_hub

In [ ]:
# 4) Drive 연결 후 KEMDy19 압축 풀기 — 경로가 다르면 zip 위치만 고치세요
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/KEMDy19 && unzip -q -o "/content/drive/MyDrive/KEMDy19.zip" -d /content/KEMDy19
!ls /content/KEMDy19 | head

In [ ]:
# 5) 학습 전에 데이터 확인 — 라벨이 붙은 발화 수와 라벨별 개수, 사용한 라벨 열이 나옵니다
#    '라벨이 붙은 wav를 거의 못 찾았어요'가 나오면 위 폴더 구조(annotation csv, wav)를 확인하세요
!python training/finetune.py --data /content/KEMDy19 --inspect

In [ ]:
# 6) 학습 (T4 기준 에폭당 십수 분 안팎). '[시작 전]'과 '[에폭 N]'의 macro-F1을 비교하세요
!python training/finetune.py --data /content/KEMDy19 --out /content/out --epochs 4

In [ ]:
# 7) 결과를 Drive에 저장 (런타임이 끊겨도 남도록)
!cp /content/out/hubert_emotion_kemdy19.ckpt /content/drive/MyDrive/hubert_emotion_kemdy19.ckpt
!ls -la /content/drive/MyDrive/hubert_emotion_kemdy19.ckpt

## 끝난 뒤
1. Drive의 `hubert_emotion_kemdy19.ckpt`를 PC로 내려받습니다.
2. 서버를 켤 때 `EMOTION_CKPT` 환경변수로 그 파일 경로를 알려 주면 새 모델을 씁니다.
   `set EMOTION_CKPT=C:\경로\hubert_emotion_kemdy19.ckpt` 후 `python -m uvicorn server:app --port 8000` (server 폴더에서)
3. **개선됐는지는 '[시작 전]'보다 '[에폭 N]'의 macro-F1이 오르는지로 판단하세요.** 오르지 않으면 쓰지 마세요.